# Qwen Image 2.1 — Setup

Notebook pensato per **Google Colab con A100**.

**Nuova sessione Colab:**
1. seleziona una GPU A100 (e RAM elevata se disponibile);
2. esegui la cella **Installazione**;
3. fai **Runtime → Riavvia sessione**;
4. riparti dalla cella **Carica Qwen Image 2.1**.

Non aggiornare manualmente Torch/CUDA.

In [ ]:
# Controllo GPU (opzionale)
!nvidia-smi

In [ ]:
# Installazione librerie
!pip install -q -U "transformers>=5.17,<5.18" accelerate pillow
!pip install -q -U git+https://github.com/huggingface/diffusers.git
!pip uninstall -y torchao

print("✅ Installazione completata.")
print("Ora: Runtime → Riavvia sessione, poi riparti dalla cella successiva.")

## Carica Qwen Image 2.1

Esegui questa cella **dopo il riavvio della sessione**.

In [ ]:
import torch
from diffusers import QwenImage21Pipeline

pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
).to("cuda")

print("✅ Pipeline caricata su GPU")
print("GPU:", torch.cuda.get_device_name(0))

# Text-to-Image

Generazione di immagini **da zero**, senza immagine di partenza.

Gli output vengono salvati in nuove cartelle timestamp dentro:

`MyDrive/QwenT2I/output`

In [ ]:
# Collega Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cartella output
import os

BASE_DIR = "/content/drive/MyDrive/QwenT2I"
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📤 Output:", OUTPUT_DIR)

## Prompt, formato e qualità

`PRESET = "1K"` è ideale per prove veloci.  
`PRESET = "2K"` usa le risoluzioni raccomandate ufficialmente per Qwen Image 2.1.

Formati disponibili: `1:1`, `4:3`, `3:4`, `3:2`, `2:3`, `16:9`, `9:16`.

In [ ]:
PROMPT = '''
A futuristic luxury wellness center overlooking the Mediterranean Sea at sunset,
premium Italian interior design, cinematic yet realistic lighting,
photorealistic commercial photography, exceptional materials and details.
'''

STEPS = 30
SEED = 42
VARIANTS = 1

PRESET = "1K"       # "1K" oppure "2K"
ASPECT_RATIO = "16:9"

SIZES = {
    "1K": {
        "1:1":  (1024, 1024),
        "4:3":  (1200, 896),
        "3:4":  (896, 1200),
        "3:2":  (1264, 848),
        "2:3":  (848, 1264),
        "16:9": (1376, 768),
        "9:16": (768, 1376),
    },
    "2K": {
        "1:1":  (2048, 2048),
        "4:3":  (2400, 1792),
        "3:4":  (1792, 2400),
        "3:2":  (2528, 1696),
        "2:3":  (1696, 2528),
        "16:9": (2752, 1536),
        "9:16": (1536, 2752),
    },
}

WIDTH, HEIGHT = SIZES[PRESET][ASPECT_RATIO]

print("✅ Prompt pronto")
print(f"Formato: {ASPECT_RATIO} — {WIDTH}×{HEIGHT}")
print("Steps:", STEPS, "| Seed iniziale:", SEED, "| Variants:", VARIANTS)

In [ ]:
# Generazione
import os
import json
import torch
from datetime import datetime

if "pipe" not in globals():
    raise RuntimeError("❌ Devi prima caricare Qwen Image 2.1.")

run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, run_id)
os.makedirs(RUN_OUTPUT_DIR, exist_ok=True)

with open(os.path.join(RUN_OUTPUT_DIR, "prompt.txt"), "w", encoding="utf-8") as f:
    f.write(PROMPT.strip())

with open(os.path.join(RUN_OUTPUT_DIR, "settings.json"), "w", encoding="utf-8") as f:
    json.dump({
        "steps": STEPS,
        "seed": SEED,
        "variants": VARIANTS,
        "preset": PRESET,
        "aspect_ratio": ASPECT_RATIO,
        "width": WIDTH,
        "height": HEIGHT,
    }, f, indent=2)

for i in range(VARIANTS):
    current_seed = SEED + i
    print(f"🎨 Variante {i+1}/{VARIANTS} — seed {current_seed}")

    image = pipe(
        prompt=PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        generator=torch.Generator("cuda").manual_seed(current_seed),
    ).images[0]

    output_path = os.path.join(RUN_OUTPUT_DIR, f"qwen_seed_{current_seed}.png")
    image.save(output_path)
    print("✅ Salvata:", output_path)

print("\n🎉 Generazione completata.")